# Lesson 5 — Budgets

**You will:** see why an unbounded agent is dangerous, then watch a budget save it.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/05-budgets/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

An agent loop has no natural stopping point. The model decides when to stop by emitting a final text response. A confused, broken, or adversarially-prompted model can keep calling tools forever. Without bounds, it can:

- Burn through your API credits in minutes.
- Hammer external services.
- Hang your application indefinitely.

**Set bounds in advance.** Step limits, tool-call limits, token limits, dollar limits. When any limit is hit, the run stops cleanly with `status='budget_exceeded'`.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")
print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")

## A trap: a tool that never finds anything

This tool always says 'try again'. Without a budget, the agent could call it forever. Watch the budget catch it.

In [ ]:
from barebear import Bear, Task, Tool, Policy, OpenRouterModel

def keep_searching(query: str) -> str:
    return "Found nothing useful. Try a different search."   # the trap

tight_policy = Policy(max_steps=4, max_cost_usd=0.01)

bear = Bear(
    model=OpenRouterModel(),
    tools=[Tool(name="keep_searching", fn=keep_searching, description="Search the web")],
    policy=tight_policy,
)

report = bear.run(Task(goal="Find the answer to a hard, ambiguous question."), trace=True)
print()
print("status:", report.status)         # almost certainly 'budget_exceeded'
print("steps used:", len(report.steps))
print("cost:", report.total_cost_usd)

## Read the receipt

The run stopped. Cleanly. No infinite loop. The `Report` tells you *exactly* how much it consumed:

```python
print(report.summary())
```

Production agents need this. You set the budget *before* the run, and the framework holds the line.

In [ ]:
print(report.summary())

## Exercise

1. Comment out the `Policy` argument and re-run. **Don't leave it running** — interrupt the cell after 30 seconds.
2. Add `max_tool_calls=2` while keeping `max_steps=4`. Predict which limit trips first.

In [ ]:
# Your turn — edit and run.

## What's next

[Lesson 6 →](../06-policy-and-guardrails/lesson.ipynb)